# Level 3 – Task 2: NLP – Text Classification (Sentiment Analysis)
**Dataset:** Social Media Sentiment Dataset
**Goal:** Classify tweets/posts as Positive, Negative, or Neutral using NLP techniques.


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)

sns.set_style("whitegrid")

df = pd.read_csv('../3) Sentiment dataset.csv')
print("Shape:", df.shape)
df[['Text', 'Sentiment']].head()


Shape: (732, 15)


,Text,Sentiment
0,Enjoying a beautiful day at the park! ...,Positive
1,Traffic was terrible this morning. ...,Negative
2,Just finished an amazing workout! 💪 ...,Positive
3,Excited about the upcoming weekend getaway! ...,Positive
4,Trying out a new recipe for dinner tonight. ...,Neutral


## 1. Explore the Dataset

In [2]:
print("Sentiment distribution:")
print(df['Sentiment'].value_counts())
print()

plt.figure(figsize=(7, 4))
df['Sentiment'].value_counts().plot(kind='bar', color=['green', 'red', 'grey'], edgecolor='black')
plt.title("Sentiment Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


Sentiment distribution:
Sentiment
Positive               44
Joy                    42
Excitement             32
Happy                  14
Neutral                14
                       ..
Vibrancy                1
Culinary Adventure      1
Mesmerizing             1
Thrilling Journey       1
Winter Magic            1
Name: count, Length: 279, dtype: int64



In [3]:
# Sample texts from each class
for sentiment in df['Sentiment'].unique():
    sample = df[df['Sentiment'] == sentiment]['Text'].iloc[0]
    print(f"[{sentiment}]: {sample[:120]}")
    print()


[ Positive  ]:  Enjoying a beautiful day at the park!              

[ Negative  ]:  Traffic was terrible this morning.                 

[ Neutral   ]:  Trying out a new recipe for dinner tonight.        

[ Anger        ]:  Can't believe the injustice happening in our society.

[ Fear         ]:  Feeling a sense of fear after watching a thriller movie. 

[ Sadness      ]:  Heartbroken after hearing the news about a natural disaster. 

[ Disgust      ]:  The state of the world's environment is just disgusting. 

[ Happiness    ]:  Pure happiness: celebrating a loved one's achievement! 

[ Joy          ]:  Laughter is the best medicine—enjoying a comedy show. 

[ Love         ]:  Sharing love and positive vibes with everyone! ❤️      

[ Amusement    ]:  An amusing incident brightened up my day!               

[ Enjoyment    ]:  Enjoying a quiet evening with a book and some tea.      

[ Admiration   ]:  Admiring the beauty of nature during a peaceful hike.   

[ Affection    ]:  Send

## 2. Text Preprocessing

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['Text'].apply(clean_text)

print("Original:  ", df['Text'].iloc[0])
print("Cleaned:   ", df['clean_text'].iloc[0])


Original:    Enjoying a beautiful day at the park!              
Cleaned:    enjoying beautiful day park


## 3. TF-IDF Vectorization

In [5]:
# TF-IDF converts text to numbers – each word gets a score based on
# how often it appears in a document vs how rare it is across all documents
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X = tfidf.fit_transform(df['clean_text'])
y = df['Sentiment']

print("Feature matrix shape:", X.shape)
print("Classes:", y.unique())


Feature matrix shape: (732, 5000)
Classes: [' Positive  ' ' Negative  ' ' Neutral   ' ' Anger        '
 ' Fear         ' ' Sadness      ' ' Disgust      ' ' Happiness    '
 ' Joy          ' ' Love         ' ' Amusement    ' ' Enjoyment    '
 ' Admiration   ' ' Affection    ' ' Awe          ' ' Disappointed '
 ' Surprise     ' ' Acceptance   ' ' Adoration    ' ' Anticipation '
 ' Bitter       ' ' Calmness     ' ' Confusion    ' ' Excitement   '
 ' Kind         ' ' Pride        ' ' Shame        ' ' Confusion '
 ' Excitement ' ' Shame ' ' Elation       ' ' Euphoria      '
 ' Contentment   ' ' Serenity      ' ' Gratitude     ' ' Hope          '
 ' Empowerment   ' ' Compassion    ' ' Tenderness    ' ' Arousal       '
 ' Enthusiasm    ' ' Fulfillment  ' ' Reverence     ' ' Compassion'
 ' Fulfillment   ' ' Reverence ' ' Elation   ' ' Despair         '
 ' Grief           ' ' Loneliness      ' ' Jealousy        '
 ' Resentment      ' ' Frustration     ' ' Boredom         '
 ' Anxiety         ' 

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

Train: 585  |  Test: 147


## 4. Train Naive Bayes

In [7]:
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print()
print(classification_report(y_test, nb_pred))


Naive Bayes Accuracy: 0.09523809523809523

                        precision    recall  f1-score   support

         Acceptance          0.00      0.00      0.00         2
           Admiration        0.00      0.00      0.00         1
        Admiration           0.00      0.00      0.00         1
         Affection           0.00      0.00      0.00         1
      Ambivalence            0.00      0.00      0.00         1
         Anger               0.00      0.00      0.00         1
        Anticipation         0.00      0.00      0.00         1
        Arousal              0.00      0.00      0.00         3
                  Awe        0.00      0.00      0.00         1
         Awe                 0.00      0.00      0.00         1
                  Bad        0.00      0.00      0.00         1
             Betrayal        0.00      0.00      0.00         2
        Betrayal             0.00      0.00      0.00         1
         Bitter              0.00      0.00      0.00       

## 5. Train Logistic Regression

In [8]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))
print()
print(classification_report(y_test, lr_pred))


Logistic Regression Accuracy: 0.09523809523809523

                        precision    recall  f1-score   support

         Acceptance          0.00      0.00      0.00         2
           Admiration        0.00      0.00      0.00         1
        Admiration           0.00      0.00      0.00         1
         Affection           0.00      0.00      0.00         1
      Ambivalence            0.00      0.00      0.00         1
         Anger               0.00      0.00      0.00         1
        Anticipation         0.00      0.00      0.00         1
        Arousal              0.00      0.00      0.00         3
                  Awe        0.00      0.00      0.00         1
         Awe                 0.00      0.00      0.00         1
                  Bad        0.00      0.00      0.00         1
             Betrayal        0.00      0.00      0.00         2
        Betrayal             0.00      0.00      0.00         1
         Bitter              0.00      0.00      0.0

## 6. Compare Models

In [9]:
results = pd.DataFrame({
    'Model':    ['Naive Bayes', 'Logistic Regression'],
    'Accuracy': [accuracy_score(y_test, nb_pred), accuracy_score(y_test, lr_pred)],
    'F1 (weighted)': [
        f1_score(y_test, nb_pred, average='weighted'),
        f1_score(y_test, lr_pred, average='weighted')
    ]
}).round(4)

print(results.to_string(index=False))

plt.figure(figsize=(7, 4))
results.set_index('Model')['Accuracy'].plot(kind='bar', color=['steelblue', 'salmon'],
                                             edgecolor='black')
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


              Model  Accuracy  F1 (weighted)
        Naive Bayes    0.0952         0.0277
Logistic Regression    0.0952         0.0273


## 7. Confusion Matrix

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

classes = sorted(y.unique())

for ax, (name, pred) in zip([ax1, ax2], [('Naive Bayes', nb_pred),
                                          ('Logistic Regression', lr_pred)]):
    cm = confusion_matrix(y_test, pred, labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(name)
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")

plt.suptitle("Confusion Matrices - Sentiment Classification", fontsize=13)
plt.tight_layout()
plt.tight_layout()
fig


## 8. Top Words Per Sentiment Class

In [11]:
# Which words drive each sentiment prediction in Logistic Regression?
feature_names = tfidf.get_feature_names_out()

for i, cls in enumerate(lr.classes_):
    top_indices = np.argsort(lr.coef_[i])[-10:][::-1]
    top_words = [feature_names[j] for j in top_indices]
    print(f"[{cls}] top words: {', '.join(top_words)}")


[ Acceptance   ] top words: diversity world, diversity, reflecting beauty, reflecting, beauty, world, young every, abyss time, bookstore, young
[ Acceptance      ] top words: finding acceptance, acceptance, finding, embracing, imperfection, imperfection finding, life, acceptance mosaic, mosaic life, mosaic
[ Accomplishment ] top words: accomplishment, glow accomplishment, milestone stepping, sense accomplishment, challenging workout, workout, completing challenging, basking, stone, completing
[ Admiration   ] top words: admiring, dedication volunteer, dedication, hike, nature peaceful, peaceful hike, local charity, charity, peaceful, local
[ Adoration    ] top words: adoration, overflowing adoration, overflowing, pet, cute rescue, rescue puppy, rescue, puppy, cute, admiration
[ Adrenaline     ] top words: rush rollercoasters, rollercoasters wild, riding adrenaline, riding, rush, adrenaline, wild, rollercoasters, twist, young
[ Adventure ] top words: dance majesty, majesty, majesty snow

## Summary
- Preprocessed text: lowercased, removed noise, lemmatized, removed stopwords.
- Converted text to TF-IDF vectors (bigrams included for context).
- **Logistic Regression outperforms Naive Bayes** on this dataset.
- The top predictive words per sentiment class align well with intuition.
